In [12]:
!pip install sentence-transformers faiss-cpu requests


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import requests

C:\Users\siddh\TruthLens\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus size:", len(corpus))

Corpus size: 20897


In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4062.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
corpus_embeddings = model.encode(corpus, show_progress_bar=True)

Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 654/654 [01:41<00:00,  6.47it/s]


In [5]:
dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(corpus_embeddings))

print("FAISS index built:", index.ntotal)

FAISS index built: 20897


In [6]:
faiss.write_index(index, "faiss_index.index")

In [7]:
def web_search(query, max_results=5):

    keywords = " ".join(query.split()[:6])
    query = keywords + " news"

    url = "https://api.duckduckgo.com/"

    params = {
        "q": query,
        "format": "json",
        "no_html": 1
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
    except:
        return []

    results = []

    if data.get("AbstractText"):
        results.append(data["AbstractText"])

    for topic in data.get("RelatedTopics", []):
        if isinstance(topic, dict) and topic.get("Text"):
            results.append(topic["Text"])

    return results[:max_results]

In [8]:
def web_search_mock(query):
    return [
        "A recent industrial accident disrupted supply chains globally.",
        "Reports indicate a refinery explosion affected fuel availability.",
        "Government advisory issued to prevent panic buying."
    ]

In [9]:
def clean_web_results(results):

    cleaned = []

    for r in results:
        if len(r.split()) < 5:
            continue
        cleaned.append(r.strip())

    return list(set(cleaned))

In [10]:
import json

def add_to_corpus(new_texts):

    global corpus, index

    # 🔥 remove duplicates
    new_texts = [t for t in new_texts if t not in corpus]

    if not new_texts:
        print("⚠️ No unique data to add")
        return

    before = len(corpus)

    corpus.extend(new_texts)

    # 🔥 UPDATE FAISS
    new_embeddings = model.encode(new_texts)
    index.add(np.array(new_embeddings))

    after = len(corpus)

    print(f"Added {after - before} new entries")
    print(f"Corpus size: {before} → {after}")
    print(f"FAISS size: {index.ntotal}")

    # 🔥 SAVE TO FILE (IMPORTANT)
    with open("corpus.json", "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)

    print("💾 Corpus saved to file")

In [11]:
def hybrid_retrieve(claim, top_k=5, threshold=0.65, use_mock=False):

    claim_embedding = model.encode([claim])

    distances, indices = index.search(np.array(claim_embedding), top_k)

    similarity = 1 / (1 + distances[0][0])

    local_results = [corpus[i] for i in indices[0]]

    print("\n==============================")
    print("CLAIM:", claim)
    print("Similarity:", round(similarity, 3))

    # ✅ GOOD MATCH
    if similarity >= threshold:
        print("Using FAISS corpus")
        return local_results

    # 🌐 FALLBACK
    print("Using WEB fallback")
    
    web_results = web_search_serpapi(claim)

    web_results = clean_web_results(web_results)

    print("Web results:", len(web_results))

    add_to_corpus(web_results)

    return web_results if web_results else local_results

In [12]:
claims = [
    "7th term in fibonacci series is 13 and it is greater than 10"

]

In [15]:
for claim in claims:
    results = hybrid_retrieve(claim, use_mock=False)

    print("\nEvidence:")
    for r in results:
        print("→", r)


CLAIM: 7th term in fibonacci series is 13 and it is greater than 10
Similarity: 0.422
Using WEB fallback
Web results: 5
Added 5 new entries
Corpus size: 20897 → 20902
FAISS size: 20902
💾 Corpus saved to file

Evidence:
→ Every Fibonacci number bigger than 1 [except F(6)=8 and F(12)=144] has at least one prime factor that is not a factor of any earlier Fibonacci number. Those ...
→ In mathematics, the Fibonacci sequence is a sequence in which each element is the sum of the two elements that precede it.
→ Where φ is the Golden ratio, approximately 1.618 F(n) = (φ^(n)-(1-φ)^(n))/√5 F(7) = (φ^(7)-(1-φ)^(7))/√5 = 13 QED. 2y.
→ The Fibonacci sequence has this form: Fn + 2 = Fn + 1 + Fn with these starting values F0 = 0 and F1 = 1. Find the 7th term via similar ...
→ The Fibonacci Sequence is the series of numbers: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ... The next number is found by adding up the two numbers before it.


In [16]:
print("\nFinal corpus size:", len(corpus))
print("FAISS size:", index.ntotal)

print("\nLast entries:")
for c in corpus[-5:]:
    print("→", c)


Final corpus size: 20902
FAISS size: 20902

Last entries:
→ Every Fibonacci number bigger than 1 [except F(6)=8 and F(12)=144] has at least one prime factor that is not a factor of any earlier Fibonacci number. Those ...
→ In mathematics, the Fibonacci sequence is a sequence in which each element is the sum of the two elements that precede it.
→ Where φ is the Golden ratio, approximately 1.618 F(n) = (φ^(n)-(1-φ)^(n))/√5 F(7) = (φ^(7)-(1-φ)^(7))/√5 = 13 QED. 2y.
→ The Fibonacci sequence has this form: Fn + 2 = Fn + 1 + Fn with these starting values F0 = 0 and F1 = 1. Find the 7th term via similar ...
→ The Fibonacci Sequence is the series of numbers: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ... The next number is found by adding up the two numbers before it.


In [25]:
!pip install google-search-results

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for google-search-results: started
  Running setup.py install for google-search-results: finished with status 'done'


  DEPRECATION: google-search-results is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from serpapi import GoogleSearch

def web_search_serpapi(query, max_results=5):

    params = {
        "q": query,
        "api_key": "a1998f816e6a8c676e95c504b8b0620ecfde0eab89b4fff491169872753f4093",
        "num": max_results
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    snippets = []

    for r in results.get("organic_results", []):
        if "snippet" in r:
            snippets.append(r["snippet"])

    return snippets[:max_results]

In [34]:
import json

with open("corpus.json", "r", encoding="utf-8") as f:
    corpus_file = json.load(f)

print("Corpus size from file:", len(corpus_file))

Corpus size from file: 20882


In [1]:
print("hybrid_retrieve" in globals())

False


In [ ]:
for x in globals():
    if callable(globals()[x]):
        print(x)